In [0]:
# ============================================================================
# Avaliação de Qualidade de Dados — silver.aerodromos
# ============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

TABELA = "voebem.silver.aerodromos"
df = spark.table(TABELA)

colunas = df.columns
n_linhas = df.count()

print(f"📋 Tabela: {TABELA}")
print(f"📊 Total de linhas: {n_linhas:,}")
print(f"🗂️  Total de colunas: {len(colunas)}")

# ---------------------------------------------------------------------------
# 1. ESQUEMA — tipos e metadados
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("1. ESQUEMA (Schema)")
print("="*70)
df.printSchema()

# ---------------------------------------------------------------------------
# 2. NULOS / AUSENTES por coluna
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("2. ANÁLISE DE NULOS / VALORES AUSENTES")
print("="*70)

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas
]).collect()[0]

null_info = []
for c in colunas:
    n_null = null_counts[c]
    pct = (n_null / n_linhas * 100) if n_linhas > 0 else 0
    null_info.append((c, n_null, pct))

null_df = pd.DataFrame(null_info, columns=["Coluna", "Nulos", "%_Nulos"])
null_df = null_df.sort_values("%_Nulos", ascending=False)
print(null_df.to_string(index=False))

# ---------------------------------------------------------------------------
# 3. CARDINALIDADE — valores distintos por coluna
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("3. CARDINALIDADE (valores distintos)")
print("="*70)

card_info = []
for c in colunas:
    n_distinct = df.select(c).distinct().count()
    pct_distinct = (n_distinct / n_linhas * 100) if n_linhas > 0 else 0
    card_info.append((c, n_distinct, pct_distinct))

card_df = pd.DataFrame(card_info, columns=["Coluna", "Distintos", "%_Distintos"])
card_df = card_df.sort_values("Distintos", ascending=False)
print(card_df.to_string(index=False))

# ---------------------------------------------------------------------------
# 4. DUPLICATAS — linhas exatas duplicadas
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("4. DUPLICATAS (linhas inteiras)")
print("="*70)

n_dup = n_linhas - df.distinct().count()
print(f"Linhas duplicadas (exatas): {n_dup:,} ({n_dup/n_linhas*100:.2f}% do total)" if n_linhas > 0 else "Sem dados")

# ---------------------------------------------------------------------------
# 5. ESTATÍSTICAS DESCRITIVAS — colunas numéricas
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("5. ESTATÍSTICAS DESCRITIVAS (colunas numéricas)")
print("="*70)

num_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType, DecimalType))]
if num_cols:
    df.describe(num_cols).display()
else:
    print("Nenhuma coluna numérica encontrada.")

# ---------------------------------------------------------------------------
# 6. DISTRIBUIÇÃO — colunas categóricas de baixa cardinalidade
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("6. DISTRIBUIÇÃO DE COLUNAS CATEGÓRICAS (cardinalidade <= 30)")
print("="*70)

cat_cols = [c for c in colunas if card_df[card_df["Coluna"] == c]["Distintos"].values[0] <= 30]
for c in cat_cols:
    print(f"\n--- {c} ---")
    df.groupBy(c).count().orderBy(F.desc("count")).show(30, truncate=False)

# ---------------------------------------------------------------------------
# 7. STRINGS VAZIAS OU APENAS ESPAÇOS
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("7. STRINGS VAZIAS / APENAS ESPAÇOS")
print("="*70)

str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
if str_cols:
    empty_info = []
    for c in str_cols:
        n_empty = df.filter((F.col(c) == "") | (F.trim(F.col(c)) == "")).count()
        if n_empty > 0:
            empty_info.append((c, n_empty))
    if empty_info:
        empty_df = pd.DataFrame(empty_info, columns=["Coluna", "Strings_Vazias"])
        print(empty_df.to_string(index=False))
    else:
        print("Nenhuma string vazia ou apenas espaços encontrada.")
else:
    print("Nenhuma coluna string encontrada.")

# ---------------------------------------------------------------------------
# 8. RESUMO DE QUALIDADE E RECOMENDAÇÕES
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("8. RESUMO DE QUALIDADE E TRATAMENTOS RECOMENDADOS")
print("="*70)

recomendacoes = []

# 8a. Nulos
for _, row in null_df.iterrows():
    if row["%_Nulos"] > 0:
        severidade = "🔴 CRÍTICO" if row["%_Nulos"] > 50 else ("🟡 ATENÇÃO" if row["%_Nulos"] > 10 else "🟢 BAIXO")
        recomendacoes.append(
            f"{severidade} — Nulos em '{row['Coluna']}': {row['Nulos']:,.0f} ({row['%_Nulos']:.1f}%). "
            f"Tratamento: preencher com valor padrão, aplicar imputação ou definir regra de negócio."
        )

# 8b. Duplicatas
if n_dup > 0:
    recomendacoes.append(
        f"🔴 CRÍTICO — {n_dup:,} linhas duplicadas ({n_dup/n_linhas*100:.2f}%). "
        f"Tratamento: aplicar DISTINCT ou DROP DUPLICATES, investigar causa na origem."
    )

# 8c. Strings vazias
if 'empty_info' in dir() and empty_info:
    recomendacoes.append(
        f"🟡 ATENÇÃO — Strings vazias detectadas em {len(empty_info)} coluna(s). "
        f"Tratamento: normalizar para NULL ou valor padrão definido pelo negócio."
    )

# 8d. Cardinalidade
chave_candidatas = card_df[card_df["%_Distintos"] > 95]["Coluna"].tolist()
if chave_candidatas:
    recomendacoes.append(
        f"ℹ️  Colunas com alta cardinalidade (>95% distintos): {chave_candidatas}. "
        f"Avaliar se são chaves candidatas; verificar unicidade e integridade referencial."
    )

# 8e. Colunas constantes
constantes = card_df[card_df["Distintos"] == 1]["Coluna"].tolist()
if constantes:
    recomendacoes.append(
        f"ℹ️  Colunas constantes (1 valor único): {constantes}. "
        f"Avaliar necessidade de manutenção; podem ser removidas ou servir como metadados."
    )

if not recomendacoes:
    print("✅ Nenhum problema de qualidade detectado automaticamente. Recomenda-se validação semântica manual.")
else:
    for i, r in enumerate(recomendacoes, 1):
        print(f"  {i}. {r}")

print("\n" + "="*70)
print("Fim da avaliação.")
print("="*70)